# PubMedQA Medical RAG Project

## Authors: Aswini Sivakumar and Vishwa Gosavi

This notebook builds an end-to-end **Retrieval-Augmented Generation (RAG)** system for medical question answering using the **PubMedQA** dataset, **Weaviate** as the vector database, **Hugging Face embedding models**, and a **Gradio** interface for interactive querying.

The workflow in this notebook is:
1. Install the required libraries.
2. Import all dependencies and helper classes.
3. Set up credentials and configuration.
4. Load embedding and language models.
5. Create and manage the Weaviate collection.
6. Build the RAG pipeline.
7. Launch an interactive Gradio demo.
8. Run evaluation on the entire labeled PubMedQA set.

## Step 1: Install and Import Dependencies

This following code blocks installs the required packages and then imports the libraries used throughout the project.

The notebook relies on:
- **Transformers** for the language model and tokenizer
- **Sentence/Hub embeddings** for document representation
- **Weaviate** for vector storage and retrieval
- **LangChain** for document and prompt handling
- **Gradio** for the interactive user interface
- **Datasets** for loading PubMedQA

Note: This notebook runs on T4 runtime

In [ ]:
!pip install -q torch transformers accelerate datasets langchain langchain-core langchain-huggingface langchain-community weaviate-client gradio sentence-transformers

In [ ]:
import os
import torch
import gradio as gr
import pandas as pd
from typing import List
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import MetadataQuery

from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuration & Credentials

## Step 2: Load Access Tokens

This section retrieves the Hugging Face token from Colab secrets so gated or authenticated models can be loaded securely during runtime.

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HUGGINGFACE_TOKEN')

## Step 3: Configure External Services

Here the notebook defines the connection details for the Weaviate cloud instance that will store embedded PubMedQA documents and support semantic retrieval.

In [ ]:
WEAVIATE_URL = "<WEAVIATE_URL>"                       #Sample: "https://******.c0.us-west3.gcp.weaviate.cloud"
WEAVIATE_API_KEY = "<YOUR_WEAVIATE_API_KEY>"          #Sample: "1234abcd-5678-efgh-9012-ijklmnopqrst"

## Step 4: Initialize Global State

The application uses a shared `system_state` dictionary to keep track of the active embedding model, LLM, vector database manager, RAG pipeline, and collection name across Gradio interactions.


In [ ]:
# Global state for Gradio
system_state = {
    "weaviate_manager": None,
    "embeddings_model": None,
    "llm_model": None,
    "pipeline": None,
    "collection_name": None
}

# Models

## Step 5: Define the Embedding Model

This helper function initializes the embedding model used to convert medical documents and user queries into dense vector representations.

Embedding normalization is enabled so similarity search behaves more consistently during retrieval.

In [ ]:
def get_embed_model(model_name):
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={'device': 'cuda'},
        encode_kwargs={'normalize_embeddings': True, 'batch_size': 32}
    )

## Step 6: Define the Language Model

This function loads the text generation model used to produce answers from retrieved evidence.

The setup is optimized for TPU usage and handles padding/tokenizer configuration so generation works reliably in the downstream RAG pipeline.

In [ ]:
def get_llm_model(model_name):
    """LLM optimized for T4 GPU (fp16, SDPA attention)"""
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
    tokenizer.padding_side = 'left'

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        token=HF_TOKEN,
        device_map="auto",
        torch_dtype=torch.float16, # T4 optimized (better than bfloat16 here)
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        attn_implementation="sdpa", # T4 compatible alternative to flash_attention_2
    )
    model.eval()

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=3,
        temperature=0.1,
        do_sample=False, # Deterministic for Yes/No
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.2,
        return_full_text=False,
    )

    llm = HuggingFacePipeline(pipeline=pipe)
    return llm

# Weaviate & Data Management

## Step 7: Create the Vector Database Manager

The `WeaviateManager` class is responsible for:
- connecting to the Weaviate cloud instance
- creating or resetting the collection
- inserting PubMedQA documents
- running semantic search over indexed medical content

This class forms the retrieval backbone of the project.

In [ ]:
class WeaviateManager:
    def __init__(self, collection_name: str):
        self.collection_name = collection_name
        self.client = weaviate.connect_to_weaviate_cloud(
            cluster_url=WEAVIATE_URL,
            auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
            skip_init_checks=True
        )

    def create_schema(self):
        """Creates schema uniquely named after the embedding model to prevent dimension clashes"""
        if self.client.collections.exists(self.collection_name):
            print(f"Schema {self.collection_name} already exists.")
            return

        self.client.collections.create(
            self.collection_name,
            # We use 'none' because we are passing vectors locally via LangChain
            vectorizer_config=Configure.Vectorizer.none(),
            properties=[
                Property(name="metadata", data_type=DataType.TEXT),
                Property(name="content", data_type=DataType.TEXT)
            ]
        )
        print(f"Schema {self.collection_name} created successfully.")

    def add_documents(self, documents: List[Document], embeddings_model):
        collection = self.client.collections.get(self.collection_name)
        with collection.batch.fixed_size(batch_size=100) as batch:
            for doc in documents:
                vector = embeddings_model.embed_query(doc.page_content)
                batch.add_object(
                    properties={
                        "content": doc.page_content,
                        "metadata": doc.metadata.get("source", "")
                    },
                    vector=vector
                )
        if collection.batch.failed_objects:
            print(f"Failed to import {len(collection.batch.failed_objects)} docs.")

    def search_documents(self, query, embeddings_model, limit=3):
        query_embedding = embeddings_model.embed_query(query)
        collection = self.client.collections.get(self.collection_name)

        results = collection.query.hybrid(
            query=query,
            vector=query_embedding,
            limit=limit,
            alpha=0.5, # Balanced Hybrid search
            return_metadata=MetadataQuery(score=True, distance=True),
        )

        documents =[]
        for result in results.objects:
            doc = Document(
                page_content=result.properties.get("content", ""),
                metadata={
                    "source": result.properties.get("metadata", ""),
                    "similarity_score": result.metadata.score if result.metadata.score else (1 - result.metadata.distance)
                }
            )
            documents.append(doc)
        return documents

## Step 8: Build the RAG Pipeline

This block combines retrieval and generation into a single pipeline.

Process:
1. Retrieve the most relevant (top-3) medical documents from Weaviate.
2. Concatenate the retrieved context.
3. Insert the context and question into a prompt template.
4. Generate an answer with the selected LLM.

This is the main reasoning path used when a user submits a medical question in the interface.


In [ ]:
def build_rag_pipeline(weaviate_manager, embeddings_model, llm_model):
    def retriever(query: str) -> List[Document]:
        return weaviate_manager.search_documents(query, embeddings_model)

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    template = """You are a medical expert assistant. Answer ONLY with one word: Yes, No, or Maybe.
INSTRUCTIONS:
- Read the medical abstracts below
- Answer the question with ONLY ONE WORD
- Your answer must be exactly: Yes, No, or Maybe

ABSTRACTS:
{context}

QUESTION:
{question}

ANSWER:"""

    prompt = ChatPromptTemplate.from_template(template)
    setup_and_retrieval = RunnableParallel(
        context=lambda x: format_docs(retriever(x)),
        question=RunnablePassthrough()
    )

    return (setup_and_retrieval | prompt | llm_model | StrOutputParser())

# Gradio UI Functions

## Step 9: Define Gradio Backend Functions

These functions power the interactive application.

They handle tasks such as:
- initializing the models and vector database
- loading a sample of PubMedQA into Weaviate
- retrieving evidence and generating an answer for a user query

In [ ]:
def init_system(embed_choice, llm_choice):
    try:
        # Sanitize name for Weaviate Collection (no slashes/dashes)
        safe_name = embed_choice.replace("/", "_").replace("-", "_").replace(".", "_")
        col_name = f"PubMedQA_{safe_name}"

        system_state["embeddings_model"] = get_embed_model(embed_choice)
        system_state["llm_model"] = get_llm_model(llm_choice)
        system_state["weaviate_manager"] = WeaviateManager(col_name)
        system_state["weaviate_manager"].create_schema()
        system_state["pipeline"] = build_rag_pipeline(
            system_state["weaviate_manager"],
            system_state["embeddings_model"],
            system_state["llm_model"]
        )
        system_state["collection_name"] = col_name
        return f" System Initialized Successfully!\nEmbedder: {embed_choice}\nLLM: {llm_choice}\nDatabase: {col_name}"
    except Exception as e:
        return f"Error initializing: {str(e)}"

### Load and Index PubMedQA Data

This function loads the PubMed abstracts. For simplicity, the contexts from the labeled PubMedQA training split is inserted into the vector database.


In [ ]:
def load_pubmed_data():
    if not system_state["weaviate_manager"]:
        return "Please initialize the system first!"

    try:
        dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

        # Taking a sample of 200 to prevent Colab UI from timing out during demo
        #dataset = dataset.select(range(200))

        documents =[]
        for i, item in enumerate(dataset):
            context_text = " ".join(item["context"]["contexts"])
            if context_text.strip():
                documents.append(Document(
                    page_content=context_text,
                    metadata={"source": f"PubMedQA_{i}", "question": item["question"]}
                ))

        splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=100)
        split_docs = splitter.split_documents(documents)

        system_state["weaviate_manager"].add_documents(split_docs, system_state["embeddings_model"])
        return f"Successfully loaded and embedded {len(split_docs)} chunks into Weaviate!"
    except Exception as e:
        return f"Error loading data: {str(e)}"

### Run Retrieval and Answer Generation

Once the system has been initialized and data has been indexed, this function:
1. retrieves the top matching documents for the question,
2. displays the retrieved evidence, and
3. generates a final answer using the RAG pipeline.

This makes it easy to inspect both the **answer** and the **supporting context** side by side.

In [ ]:
def ask_question(query):
    if not system_state["pipeline"]:
        return "Please initialize the system first!", "System not initialized."

    # Get contexts
    docs = system_state["weaviate_manager"].search_documents(query, system_state["embeddings_model"])

    # Format contexts for display
    context_display = ""
    for i, doc in enumerate(docs, 1):
        score = doc.metadata.get('similarity_score', 0)
        context_display += f"### Document {i} (Score: {score:.4f})\n"
        context_display += f"**Source:** {doc.metadata.get('source', 'Unknown')}\n"
        context_display += f"> {doc.page_content}\n\n---\n\n"

    if not context_display:
        context_display = "No contexts retrieved. Please ensure you have loaded the PubMed data."

    # Get final answer
    answer = system_state["pipeline"].invoke(query)

    return context_display, f"## 🩺 Answer: {answer}"

# Gradio Application Layout

## Step 10: Build the Gradio Interface

This section defines the front-end layout of the project demo.

The interface allows a user to:
- choose an embedding model
- choose a language model
- initialize the system
- load PubMedQA documents
- ask a medical question
- view both the generated answer and retrieved evidence


In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧬 PubMedQA Medical RAG System")

    with gr.Row():
        with gr.Column(scale=2):
            embed_dropdown = gr.Dropdown(
                choices=["AswiniSivakumar/mini-pubmedqa-finetuned", "Qwen/Qwen3-Embedding-8B", "sentence-transformers/all-MiniLM-L6-v2"],
                value="AswiniSivakumar/mini-pubmedqa-finetuned",
                label="Embedding Model"
            )
            llm_dropdown = gr.Dropdown(
                choices=["meta-llama/Llama-3.2-1B-Instruct", "meta-llama/Llama-3.1-8B-Instruct"],
                value="meta-llama/Llama-3.2-1B-Instruct",
                label="LLM Model"
            )
        with gr.Column(scale=1):
            init_btn = gr.Button("🚀 1. Initialize Models & DB", variant="primary")
            load_data_btn = gr.Button("📚 2. Load PubMed Data to DB")
            status_box = gr.Textbox(label="System Status", lines=3, interactive=False)

    gr.Markdown("---")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🔍 Retrieved Contexts")
            contexts_output = gr.Markdown("Contexts will appear here...")

        with gr.Column(scale=1):
            gr.Markdown("### 💬 Ask a Medical Question")
            query_input = gr.Textbox(
                placeholder="e.g. Is methylation of the FGFR2 gene associated with high birth weight?",
                label="Question",
                lines=3
            )
            submit_btn = gr.Button("🧠 Get Answer", variant="primary")
            answer_output = gr.Markdown("## 🩺 Answer: ")

    # Event Bindings
    init_btn.click(init_system, inputs=[embed_dropdown, llm_dropdown], outputs=status_box)
    load_data_btn.click(load_pubmed_data, outputs=status_box)
    submit_btn.click(ask_question, inputs=query_input, outputs=[contexts_output, answer_output])

## Step 11: Launch the Application

This final command starts the Gradio app so the full RAG workflow can be tested interactively.

In [ ]:
# Launch App
demo.launch(debug=True, share=True)

## Example Medical Questions

These example prompts can be used to quickly test the system after the app is running.
- Do mutations causing low HDL-C promote increased carotid intima-media thickness?
- Aripiprazole: a new risk factor for pathological gambling?

## Evaluation Setup

The below cells run evaluation on the entire labeled PubMedQA dataset.

In [ ]:
import pandas as pd
import re
from datasets import load_dataset
from tqdm.auto import tqdm

In [ ]:
# Ensure the system is initialized and pipeline exists
if "system_state" not in globals() or system_state.get("pipeline") is None:
    print("ERROR: Please click '1. Initialize Models & DB' in the Gradio UI first!")
else:
    print("Loading the full 1000-question labeled dataset...")
    # The 'pqa_labeled' split contains exactly 1,000 expert-answered questions
    dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

    results =[]

    # Dictionary to track correct/total for each class
    metrics = {
        "yes": {"total": 0, "correct": 0},
        "no": {"total": 0, "correct": 0},
        "maybe": {"total": 0, "correct": 0}
    }

    print("Starting evaluation. This will take ~10-15 minutes on a T4 GPU...")

    # Iterate through all 1000 questions
    for item in tqdm(dataset, total=len(dataset), desc="Evaluating PubMedQA"):
        question = item["question"]
        # Ground truth is always 'yes', 'no', or 'maybe'
        ground_truth = item["final_decision"].strip().lower()

        # 1. Generate answer using our initialized RAG pipeline
        raw_answer = system_state["pipeline"].invoke(question)

        # 2. Clean the generated answer (remove punctuation, lowercase, strip whitespace)
        # E.g., "Yes." -> "yes", "\nNo" -> "no"
        clean_answer = re.sub(r'[^\w\s]', '', raw_answer).strip().lower()

        # 3. Determine if correct
        is_correct = (ground_truth == clean_answer)

        # Fallback: Just in case the 1B model was chatty (e.g., "the answer is yes")
        if not is_correct and ground_truth in clean_answer.split():
            is_correct = True

        # 4. Update metrics
        if ground_truth in metrics:
            metrics[ground_truth]["total"] += 1
            if is_correct:
                metrics[ground_truth]["correct"] += 1

        # 5. Store for the DataFrame
        results.append({
            "question": question,
            "ground_truth": ground_truth.capitalize(),
            "raw_generated_answer": raw_answer.strip(),
            "is_correct": is_correct
        })

    # --- Calculate & Display Final Results ---
    df_results = pd.DataFrame(results)

    # Save to CSV so you can inspect where the model failed
    csv_filename = "PubMedQA_1000_Eval_Results.csv"
    df_results.to_csv(csv_filename, index=False)

    total_questions = len(df_results)
    overall_correct = df_results["is_correct"].sum()
    overall_acc = (overall_correct / total_questions) * 100

    print("\n" + "="*50)
    print("FINAL EVALUATION RESULTS")
    print("="*50)
    print(f"Total Questions Evaluated : {total_questions}")
    print(f"Overall Accuracy          : {overall_acc:.2f}% ({overall_correct}/{total_questions})")
    print("-" * 50)

    # Print breakdown by class
    for category, counts in metrics.items():
        total = counts["total"]
        correct = counts["correct"]
        if total > 0:
            acc = (correct / total) * 100
            # Formatting to align the text nicely
            print(f"Class '{category.capitalize():<5}': {acc:>6.2f}% ({correct:>3}/{total:>3} correct)")
        else:
            print(f"Class '{category.capitalize():<5}': No samples found in dataset.")

    print("="*50)
    print(f"Detailed results saved to '{csv_filename}'")